# AI Step-by-Step Math Tutor — LoRA Fine-Tuning & Evaluation

This notebook runs **QLoRA fine-tuning** for `Qwen2.5-Math-1.5B-Instruct` on a free GPU (Google Colab T4/A100 or Kaggle P100/T4x2).

### Pipeline Overview
1. **Environment Setup**: Inspect GPU and install training dependencies.
2. **Smoke Test**: Train on a small 300-sample slice (1 epoch) to verify gradient updates and memory.
3. **Full Training**: Fine-tune on up to 10,000 multi-step reasoning samples.
4. **Evaluation**: Benchmark Base Model vs Fine-Tuned Adapter vs SymPy on the test set and official GSM8K.
5. **Export**: Compress and download the fine-tuned adapter weights.

## 1. Check GPU Availability & Precision Support

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Install Training Dependencies

In [ ]:
# Install training dependencies
!pip install -q --upgrade pip
!pip install -q torch transformers peft trl bitsandbytes accelerate datasets kagglehub pyyaml sympy
print("Training dependencies successfully installed.")

## 3. Clone or Copy the Project & Set Up Python Path

In [ ]:
import os
import sys

# Ensure project root is in sys.path
PROJECT_DIR = os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Check if dataset is already built; if not, build it
if not os.path.exists("tutor/data/processed/train.jsonl"):
    print("Processed dataset not found. Running build_dataset.py...")
    !python tutor/data/build_dataset.py
else:
    print("Dataset already exists in tutor/data/processed/")
    !python -c "import json; print(json.load(open('tutor/data/processed/dataset_stats.json')))"

## 4. Run Smoke Test (300 Samples, 1 Epoch)
Validates QLoRA configuration, memory headroom, dynamic evaluation steps, and loss logging.

In [ ]:
# Smoke test: 300 samples, 1 epoch
!python tutor/training/finetune_lora.py --limit 300 --epochs 1 --output-dir models/smoke-test-adapter

## 5. Run Full Training (Up to 10,000 Samples, 1 Epoch)
Trains on the complete high-quality dataset and logs progress to `tutor/training/logs/loss_log.csv`.

In [ ]:
# Full training run
!python tutor/training/finetune_lora.py --limit 10000 --epochs 1 --output-dir models/qwen2.5-math-tutor-lora

## 6. Inspect Loss Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_csv = "tutor/training/logs/loss_log.csv"
if os.path.exists(log_csv):
    df = pd.read_csv(log_csv)
    print(df.tail(10))
    
    plt.figure(figsize=(10, 4))
    if 'loss' in df.columns and df['loss'].dropna().any():
        plt.plot(df['step'], df['loss'], label='Training Loss', color='blue')
    if 'eval_loss' in df.columns and df['eval_loss'].dropna().any():
        plt.plot(df['step'], df['eval_loss'], label='Validation Loss', color='orange', marker='o')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('QLoRA Fine-Tuning Loss Curves')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.show()
else:
    print(f"Log file {log_csv} not found.")

## 7. Run Comprehensive Evaluation Benchmark
Compares:
1. **BASE**: Base model without fine-tuning
2. **FINE-TUNED**: Model with our trained LoRA adapter
3. **SYMPY**: Rule-based symbolic solver

In [ ]:
# Benchmark all 3 modes on held-out test split
!python tutor/evaluation/evaluate.py --mode all --limit 100 --adapter-path models/qwen2.5-math-tutor-lora

In [ ]:
# Benchmark on official 300-sample held-out GSM8K test split
!python tutor/evaluation/evaluate.py --mode all --gsm8k-official --limit 300 --adapter-path models/qwen2.5-math-tutor-lora

## 8. Package & Download the Adapter Weights

In [ ]:
# Zip the fine-tuned adapter directory for local download
!zip -r qwen2.5-math-tutor-lora.zip models/qwen2.5-math-tutor-lora/
print("Adapter zipped successfully as 'qwen2.5-math-tutor-lora.zip'.")

# If running in Google Colab, trigger direct browser download:
try:
    from google.colab import files
    files.download('qwen2.5-math-tutor-lora.zip')
except ImportError:
    print("To download on Kaggle, use the Output tab on the right sidebar.")